**Script to Annotate non endocrine spots**

**Author:** Yike Xie

**Date Updated:** Jan 2026

In [1]:
# Load libraries
import sys
sys.path.append("..")
import utils
import os

import numpy as np
import pandas as pd
import scanpy as sc

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [2]:
fig_folder = '../../figures/spatial/spatial_niches'
data_folder = '../../data/'
res_folder = '../../results/spatial/spatial_niches/'

os.makedirs(fig_folder, exist_ok=True)
os.makedirs(data_folder, exist_ok=True)
os.makedirs(res_folder, exist_ok=True)

# Data Loading

## basic information

In [3]:
metadata = pd.read_excel(
    '../../figures/manuscript_figures/tables/Table1_human_donor_information.xlsx',
    sheet_name='Pancreas', index_col=0)

tech_cols = ["snRNA-Seq", "Immunostaining", "Spatial transcriptomics", "Calcium imaging", "Slice-Seq"]
tech_mask = metadata[tech_cols].astype(str).apply(lambda s: s.str.lower().str.strip().eq("yes"))

metadata["Method"] = tech_mask.apply(
    lambda row: ", ".join([col for col, ok in row.items() if ok]),
    axis=1
)

## spatial data

In [ ]:
# this dataset includes the Local Moran's I results
adata = sc.read_h5ad('../../data/YK_raw_spatial_islet_extra_islet_c2l_annot_hq.h5ad')
adata.obs['group'] = metadata.loc[adata.obs['sample'].str.split('-').str[0]]['group'].to_list()

## parse scRNA-Seq data

In [ ]:
parse = sc.read_h5ad('../../data/parse_snRNA_annotated_YK_raw.h5ad')
parse = parse[parse.obs['Doublets'] == 'no']

parse_raw = parse.copy()
utils.normalizedata(parse, log1p=True)

# cell2location on non-endocrine cell subtypes

In [ ]:
adata_non_islet = adata[
    (adata.obs['islets_in_out'] != 'in') &
    (adata.obs['max_pred_celltype'] != 'Endocrine')
].copy()

## Functions

In [ ]:
import anndataks
from statsmodels.stats.multitest import multipletests

def ks(adata, cst1, cst2, log1p=2):
    
    adata1 = adata[adata.obs['cell_subtype'].isin(cst1)]
    adata2 = adata[adata.obs['cell_subtype'].isin(cst2)]
        
    if adata1.obs.shape[0] * adata2.obs.shape[0] == 0:
        return None
    else:
        res = anndataks.compare(adata1, adata2, log1p=log1p, mode='asymp') # log2fc + -- high in adata2
        res['pval_adj'] = multipletests(res["pvalue"], method="fdr_bh")[1]
        res.index.name = 'gene'
        return res

## annotation

In [ ]:
# this is based on the celllocation deconvolution on all cell subtypes except endocrines
# this result include spots with max_pred_celltype == 'Endocrine', I should manually exclude

cst_c2l_abund_path = '../../data/cell2loc_others/cell2loc_non_islet_all_nonendocrine_subtypes/cst_c2l/non_islet_c2l_merged_cst_abundance.tsv'

cst_c2l_abund = pd.read_csv(cst_c2l_abund_path, sep='\t', index_col=0)
cst_c2l_abund.columns = cst_c2l_abund.columns.str.lstrip('q05cell_abundance_w_sf_')

cst_c2l_abund = cst_c2l_abund.rename(columns={
    'Pancreatic_persistent_memory_like_T_cells': 'T_cells',
    'Pancreatic_resident_macrophages': 'Macrophages',
    'CD27-_IgA+_plasmablasts': 'Plasmablasts',
})

cst_c2l_abund['total_abund'] = cst_c2l_abund.sum(axis=1)
cst_c2l_abund['max_cst'] = cst_c2l_abund[
    cst_c2l_abund.columns[:-1]
].idxmax(axis=1)

# only keep non-endocrine spots in non-islet area
cst_c2l_abund = cst_c2l_abund.loc[adata_non_islet.obs_names]
cst_c2l_abund['max_cst'] = cst_c2l_abund['max_cst'].astype('category').cat.reorder_categories(
    [
       'Signaling_acinar', 'Intermediate_acinar', 'High_enzyme_acinar', 
        'Basal_ductal', 'Inflam_ductal_1', 'Inflam_ductal_2', 'MUC5B+_ductal', 
        'Arterial_ECs', 'Venous_ECs', 'Capillary_ECs', 'Lymphatic_ECs', 
        'Macrophages', 'Plasmablasts', 'T_cells','Mast_cells', 
        'Activated_stellates', 'Quiescent_stellates', 
        'Schwann'
    ]
)

In [ ]:
# this is based on the celllocation deconvolution on all cell types
# already excluded islet area and extra islet endocrine spots

ct_c2l_abund = adata_non_islet.obsm['q05_cell_abundance_w_sf'].copy()
ct_c2l_abund.columns = ct_c2l_abund.columns.str.lstrip('q05cell_abundance_w_sf_')

ct_c2l_abund['total_abund'] = ct_c2l_abund.sum(axis=1)
ct_c2l_abund['max_ct'] = ct_c2l_abund[
    ct_c2l_abund.columns[:-1]
].idxmax(axis=1)

ct_c2l_abund['max_ct'] = ct_c2l_abund['max_ct'].astype('category').cat.reorder_categories(
    ['Acinar', 'Ductal', 'Endothelial', 'Immune', 'Stellate', 'Schwann']
)

In [ ]:
c2l = pd.concat([ct_c2l_abund['max_ct'], cst_c2l_abund['max_cst']], axis=1, join='inner').dropna()
c2l.columns = ['max_ct','max_cst']

# 1) Column-normalized (purity per subtype)
mix_col = pd.crosstab(c2l['max_ct'], c2l['max_cst'], normalize='columns')

# 2) Row-normalized (recall per coarse type)
mix_row = pd.crosstab(c2l['max_ct'], c2l['max_cst'], normalize='index')

# 3) Fraction of all cells (global)
mix_all = pd.crosstab(c2l['max_ct'], c2l['max_cst'], normalize='all')

sns.heatmap(mix_col, vmin=0, vmax=1)   # purity per subtype
# sns.heatmap(mix_row, vmin=0, vmax=1) # recall per coarse type
# sns.heatmap(mix_all, vmin=0, vmax=1) # global fractions


In [ ]:
expected = {
    # tweak as needed
    **{c: 'Acinar'     for c in mix_col.columns if 'acinar'     in c.lower()},
    **{c: 'Ductal'     for c in mix_col.columns if 'ductal'     in c.lower()},
    **{c: 'Endothelial'for c in mix_col.columns if 'ecs'        in c.lower()},
    **{c: 'Immune'     for c in mix_col.columns if any(k in c.lower() for k in ['macroph', 'plasma', 't_cells','mast'])},
    **{c: 'Stellate'   for c in mix_col.columns if 'stellate'   in c.lower()},
    **{c: 'Schwann'    for c in mix_col.columns if 'schwann'    in c.lower()},
}

threshold = 0.7  # purity cutoff
summary = []
for col in mix_col.columns:
    exp = expected.get(col)
    maj = mix_col[col].idxmax()
    exp_share = mix_col.loc[exp, col] if exp else float('nan')
    other = mix_col[col].drop(exp).idxmax() if exp else mix_col[col].idxmax()
    summary.append({
        "subtype": col,
        "expected": exp,
        "majority": maj,
        "purity_expected": exp_share,
        "top_offtarget": other,
        "flag_mislabeled": (exp is None) or (maj != exp) or (exp_share < threshold),
    })
summary = pd.DataFrame(summary).set_index("subtype").sort_values(["flag_mislabeled","purity_expected"])
summary

In [ ]:
for ct in ['Acinar', 'Ductal', 'Endothelial', 'Immune', 'Stellate', 'Schwann']:

    pct_ct = (ct_c2l_abund[ct_c2l_abund['max_ct'] == ct]['total_abund'] < 0.5).sum() / ct_c2l_abund[ct_c2l_abund['max_ct'] == ct].shape[0]
    
    print(f'{ct} has {pct_ct * 100:.2f}% spots with < 0.5 absolute abundance')

### Ductal

In [ ]:
adata_ductal = adata_non_islet[adata_non_islet.obs['max_pred_celltype'] == 'Ductal'].copy()

ductal_csts = ['Basal_ductal', 'Inflam_ductal_1', 'Inflam_ductal_2', 'MUC5B+_ductal', ]
ductal_cst_abund = cst_c2l_abund[ductal_csts].loc[adata_ductal.obs_names].copy()
adata_ductal.obs['Ductal_max_subtype'] = ductal_cst_abund.idxmax(axis=1)

In [ ]:
ductal_genes = [ 
    "C3", 'NOSTRIN',
    
    'CREB5', 
    "RELB", "NFKB2", "IRAK2", "VCAM1", 
    "CXCL2", "CXCL1", "CCL2", "CX3CL1", "CXCL8",
    
    'MUC5B', "FOS",# high in MUC5B+ ductal cells
              ]

adata_ductal_show = adata_ductal[:, ductal_genes].copy()
sc.pp.scale(adata_ductal_show, zero_center=False)

fig, ax = plt.subplots(figsize=[7, 3])
axs = sc.pl.dotplot(adata_ductal_show, 
              var_names=ductal_genes,
              groupby="Ductal_max_subtype",
              color_map='Reds', 
              colorbar_title = 'Scaled', 
              categories_order=ductal_csts,
              use_raw=False, show=False,
              ax=ax,)

In [ ]:
adata_ductal.obs['Ductal_max_subtype'].value_counts() * 100 / adata_ductal.obs.shape[0]

In [ ]:
parse_ductal = parse[parse.obs['cell_type'] == 'Ductal'].copy()
parse_ductal.obs['cell_subtype'].value_counts() * 100 / parse_ductal.obs.shape[0]

**I would trust the annotation of inflam_ductal_1 and inflam_ductal_2, but double check MUC5B+_ductal**

In [ ]:
(adata_ductal[:, 'MUC5B'].X > 0).sum() * 100 / adata_ductal.shape[0]

In [ ]:
# find differentially expressed genes of MUC5B+ ductal cells in parse and also highly expressed in spatial data
MUC5B_res = ks(
    parse_ductal, 
    ['Basal_ductal', 'Inflam_ductal_1', 'Inflam_ductal_2'], 
    ['MUC5B+_ductal'], 
    log1p=2)

pos_genes = MUC5B_res[
    (MUC5B_res['log2_fold_change'] > 1) 
    & (MUC5B_res['pval_adj'] < 0.05)
].index.tolist()

pos_genes = adata_ductal.var_names[adata_ductal.var_names.isin(pos_genes)]
pos_gene_exp = pd.Series(
    np.asarray(adata_ductal[:, adata_ductal.var_names.isin(pos_genes)].X.mean(axis=0))[0],
    index=pos_genes
)

pos_gene_exp.sort_values(ascending=False)[:20].index

In [ ]:
# validate the expression in parse data
ductal_genes = [ 
    'SPINK1', 'PRSS3', 'FOS', 'IER2', 'ANPEP', 'RHOB', 'MUC5B', # high in MUC5B+ ductal cells
              ]

parse_ductal_show = parse_ductal[:, ductal_genes].copy()
sc.pp.scale(parse_ductal_show, zero_center=False)

fig, ax = plt.subplots(figsize=[7, 3])
axs = sc.pl.dotplot(parse_ductal_show, 
              var_names=ductal_genes,
              groupby="cell_subtype",
              color_map='Reds', 
              colorbar_title = 'Scaled', 
              categories_order=ductal_csts,
              use_raw=False, show=False,
              ax=ax,)

In [ ]:
# check the expression in spatial data
ductal_genes = [ 
    'FOS', 'IER2', 'ANPEP', 'RHOB', 'MUC5B', 
              ]

adata_ductal_show = adata_ductal[:, ductal_genes].copy()
sc.pp.scale(adata_ductal_show, zero_center=False)

fig, ax = plt.subplots(figsize=[7, 3])
axs = sc.pl.dotplot(adata_ductal_show, 
              var_names=ductal_genes,
              groupby="Ductal_max_subtype",
              color_map='Reds', 
              colorbar_title = 'Scaled', 
              categories_order=ductal_csts,
              use_raw=False, show=False,
              ax=ax,)

In [ ]:
MUC5B_genes = ['FOS', 'IER2', 'ANPEP', 'RHOB', 'MUC5B', ]

sc.tl.score_genes(
    adata_ductal,
    gene_list=MUC5B_genes,
    ctrl_size=50,        
    n_bins=25,           
    use_raw=False,       
    score_name="MUC5B_score" # signaling should have low
)

In [ ]:
fig, ax = plt.subplots(figsize=[3, 2])

ax.hist(adata_ductal.obs['MUC5B_score'], bins=100, alpha=0.5)
ax.set_ylabel('Number of beads')
ax.set_xlabel('MUC5B score')
ax.set_title('Non-islet spatial ductal')

ax.set_yscale('log')

ax.axvline(0, c='k')

In [ ]:
# make the threshold to annotate MUC5B+ ductal spots by MUC5B_score
MUC5B_score_thresh = 0 # np.quantile(adata_ductal.obs['MUC5B_score'], 0.95) if you want it by percentage

adata_ductal.obs.loc[adata_ductal.obs['Ductal_max_subtype'] == 'MUC5B+_ductal', 
                   'Ductal_max_subtype'] = 'Basal_ductal'
adata_ductal.obs.loc[adata_ductal.obs['MUC5B_score'] > MUC5B_score_thresh, 
                       'Ductal_max_subtype'] = 'MUC5B+_ductal'

print(adata_ductal.obs['Ductal_max_subtype'].value_counts() * 100 / adata_ductal.obs['Ductal_max_subtype'].shape[0])

# save the annotation
adata_ductal.obs[['Ductal_max_subtype']].to_csv(
    os.path.join(
        '../../data/cell2loc_others/cell2loc_non_islet_all_nonendocrine_subtypes/annotation_based_on_M_ct_c2l/', 
        'non_islet_c2l_ductal_annotation.tsv'
    ), sep='\t'
)

### Endothelial

In [ ]:
adata_EC = adata_non_islet[adata_non_islet.obs['max_pred_celltype'] == 'Endothelial'].copy()

EC_csts = ['Arterial_ECs', 'Venous_ECs', 'Capillary_ECs', 'Lymphatic_ECs', ]
EC_cst_abund = cst_c2l_abund[EC_csts].loc[adata_EC.obs_names].copy()
adata_EC.obs['Endothelial_max_subtype'] = EC_cst_abund.idxmax(axis=1)

In [ ]:
EC_genes = [
    'PECAM1', # marker gene
    'CDH5',  'GJA5', 'DLL4', 'BMX',# arterial
    'VWF', 'EPHB4', 'NR2F2', 'ACKR1',# venous
    'CD36',  #'PLVAP',  # capillary
    'FLT4','PROX1', 'LYVE1',  # lymphatic  
]

adata_EC_show = adata_EC[:, EC_genes].copy()
sc.pp.scale(adata_EC_show, zero_center=False)

fig, ax = plt.subplots(figsize=[7, 3])
axs = sc.pl.dotplot(adata_EC_show, 
              var_names=EC_genes,
              groupby="Endothelial_max_subtype",
              color_map='Reds', 
              colorbar_title = 'Scaled', 
              categories_order=EC_csts,
              use_raw=False, show=False,
              ax=ax,)

In [ ]:
EC_genes = [
    'PECAM1', # marker gene
    'CDH5',  'GJA5', 'DLL4', 'BMX',# arterial
    'VWF', 'EPHB4', 'NR2F2', 'ACKR1',# venous
    'CD36',  #'PLVAP',  # capillary
    'PROX1', 'LYVE1',  # lymphatic  
]

fig, ax = plt.subplots(figsize=[7, 3])
axs = sc.pl.dotplot(adata_EC_show, 
              var_names=EC_genes,
              groupby="Endothelial_max_subtype",
              color_map='Reds', 
              colorbar_title = 'Scaled', 
              categories_order=EC_csts,
              use_raw=False, show=False,
              ax=ax,)

In [ ]:
adata_EC.obs['Endothelial_max_subtype'].value_counts() * 100 / adata_EC.obs.shape[0]

In [ ]:
parse_EC = parse[parse.obs['cell_type'] == 'Endothelial'].copy()
parse_EC.obs['cell_subtype1'].value_counts() * 100 / parse_EC.obs.shape[0]

**I would trust the annotation of endothelial subtypes**

In [ ]:
# save the annotation
adata_EC.obs[['Endothelial_max_subtype']].to_csv(
    os.path.join(
        '../../data/cell2loc_others/cell2loc_non_islet_all_nonendocrine_subtypes/annotation_based_on_M_ct_c2l/', 
        'non_islet_c2l_endothelial_annotation.tsv'
    ), sep='\t'
)

### Immune

In [ ]:
adata_immune = adata_non_islet[adata_non_islet.obs['max_pred_celltype'] == 'Immune'].copy()

immune_csts = ['T_cells','Macrophages', 'Plasmablasts', 'Mast_cells', ]
immune_cst_abund = cst_c2l_abund[immune_csts].loc[adata_immune.obs_names].copy()
adata_immune.obs['Immune_max_subtype'] = immune_cst_abund.idxmax(axis=1)

In [ ]:
immune_genes = [
    "CD247",  # T cell lineage
    "IL7R", # naive or memory T cells
    "CD69", # tissue-resident memory T cells
#     "CD3E",  # low expression, suggests quiescence
#     "CD4",  "CD8A", # confirm absence of lineage commitment
    
    'CD163', # macrophage marker
    'F13A1', # tissue resident macrophages
    'ITGAX', # Inflammatory / activated macrophages and DCs
    'HLA-DRB1', # MHC II genes
    'CD14', # optional: 'CD68', 'LYZ', # Monocyte/infiltrating macrophage markers
    
    'CD38',  # plsamablasts and plasma cells
    'IGHA1', 'IGHA2',  # Immunoglobulin isotypes
    "PRDM1", "XBP1", "JCHAIN", # Core antibody-secreting transcriptional program
    "SDC1" , # CD138,  plasma cell markers
    "MZB1",  # plasma cell markers, low in plasmablasts
    'MS4A1', # CD 20, B cell markers, low in plasmablasts
    
    "IL1RL1", 'KIT', 
    "TPSAB1", 'AREG',
    "MS4A2", 
    "CD34"# mast cells
]

adata_immune_show = adata_immune[:, immune_genes].copy()
sc.pp.scale(adata_immune_show, zero_center=False)

fig, ax = plt.subplots(figsize=[7, 3])
axs = sc.pl.dotplot(adata_immune_show, 
              var_names=immune_genes,
              groupby="Immune_max_subtype",
              color_map='Reds', 
              colorbar_title = 'Scaled', 
              categories_order=immune_csts,
              use_raw=False, show=False,
              ax=ax,)

In [ ]:
fig, ax = plt.subplots(figsize=[7, 3])
axs = sc.pl.dotplot(adata_immune_show, 
              var_names=["CD247", 'IL7R', 'CD69'],
              groupby="Immune_max_subtype",
              color_map='Reds', 
              colorbar_title = 'Scaled', 
              categories_order=immune_csts,
              use_raw=False, show=False,
              ax=ax,)

In [ ]:
adata_immune.obs['Immune_max_subtype'].value_counts() * 100 / adata_immune.obs.shape[0]

In [ ]:
parse_immune = parse[parse.obs['cell_type'] == 'Immune'].copy()
parse_immune.obs['cell_subtype'].value_counts() * 100 / parse_immune.obs.shape[0]

In [ ]:
# I can trust mast cells
# Any other spots expressing HLA-DRB1 should be macrophages

non_mac = adata_immune[adata_immune.obs['Immune_max_subtype'] != 'Macrophages'].copy()
adata_immune.obs.loc[
    adata_immune.obs_names.isin(non_mac.obs_names[non_mac[:, 'HLA-DRB1'].X.toarray()[:, 0] > 0]),
    'Immune_max_subtype'] = 'Macrophages'

adata_immune.obs['Immune_max_subtype'].value_counts() * 100 / adata_immune.obs.shape[0]

adata_immune.obs[['Immune_max_subtype']].to_csv(
    os.path.join(
        '../../data/cell2loc_others/cell2loc_non_islet_all_nonendocrine_subtypes/annotation_based_on_M_ct_c2l/', 
        'non_islet_c2l_immune_annotation.tsv'
    ), sep='\t'
)

### Stellate

In [ ]:
adata_stellate = adata_non_islet[adata_non_islet.obs['max_pred_celltype'] == 'Stellate'].copy()

stellate_csts = ['Activated_stellates', 'Quiescent_stellates', ]
stellate_cst_abund = cst_c2l_abund[stellate_csts].loc[adata_stellate.obs_names].copy()
adata_stellate.obs['Stellate_max_subtype'] = stellate_cst_abund.idxmax(axis=1)

In [ ]:
stellate_genes = [
    "COL6A3", # activated stellate
    "RGS5", # quiescent stellate
]

adata_stellate_show = adata_stellate[:, stellate_genes].copy()
sc.pp.scale(adata_stellate_show, zero_center=False)

fig, ax = plt.subplots(figsize=[7, 3])
axs = sc.pl.dotplot(adata_stellate_show, 
              var_names=stellate_genes,
              groupby="Stellate_max_subtype",
              color_map='Reds', 
              colorbar_title = 'Scaled', 
              categories_order=stellate_csts,
              use_raw=False, show=False,
              ax=ax,)

In [ ]:
adata_stellate.obs['Stellate_max_subtype'].value_counts() * 100 / adata_stellate.obs.shape[0]

In [ ]:
parse_stellate = parse[parse.obs['cell_type'] == 'Stellate'].copy()
parse_stellate.obs['cell_subtype'].value_counts() * 100 / parse_stellate.obs.shape[0]

In [ ]:
adata_stellate.obs[['Stellate_max_subtype']].to_csv(
    os.path.join(
        '../../data/cell2loc_others/cell2loc_non_islet_all_nonendocrine_subtypes/annotation_based_on_M_ct_c2l/', 
        'non_islet_c2l_stellate_annotation.tsv'
    ), sep='\t'
)

### summary

In [ ]:
adata_non_islet.obs['max_pred_subtype'] = np.nan

adata_non_islet.obs.loc[adata_non_islet.obs['max_pred_celltype'] == 'Acinar', 'max_pred_subtype'] = 'Acinar'
adata_non_islet.obs.loc[adata_non_islet.obs['max_pred_celltype'] == 'Schwann', 'max_pred_subtype'] = 'Schwann'
adata_non_islet.obs.loc[adata_ductal.obs_names, 'max_pred_subtype'] = adata_ductal.obs['Ductal_max_subtype']
adata_non_islet.obs.loc[adata_immune.obs_names, 'max_pred_subtype'] = adata_immune.obs['Immune_max_subtype']
adata_non_islet.obs.loc[adata_EC.obs_names, 'max_pred_subtype'] = adata_EC.obs['Endothelial_max_subtype']
adata_non_islet.obs.loc[adata_stellate.obs_names, 'max_pred_subtype'] = adata_stellate.obs['Stellate_max_subtype']

adata_non_islet.obs['max_pred_subtype'].value_counts()

In [ ]:
adata_non_islet.obs['max_pred_celltype'] = adata_non_islet.obs['max_pred_celltype'].astype('category').cat.reorder_categories(
    ['Acinar', 'Ductal', 'Endothelial', 'Immune', 'Stellate', 'Schwann']
)
adata_non_islet.obs['max_pred_subtype'] = adata_non_islet.obs['max_pred_subtype'].astype('category').cat.reorder_categories(
    ['Acinar', 
     'Basal_ductal', 'Inflam_ductal_1', 'Inflam_ductal_2', 'MUC5B+_ductal', 
     'Arterial_ECs', 'Venous_ECs', 'Capillary_ECs', 'Lymphatic_ECs', 
     'Macrophages', 'Plasmablasts', 'T_cells', 'Mast_cells', 
     'Activated_stellates', 'Quiescent_stellates', 
     'Schwann'
    ]
)

# save the annotation
adata_non_islet.obs[['max_pred_subtype']].to_csv(
    os.path.join(
        '../../data/cell2loc_others/cell2loc_non_islet_all_nonendocrine_subtypes/annotation_based_on_M_ct_c2l/', 
        'non_islet_adata_YK_cellsubtype_annotation.tsv'
    ), sep='\t'
)

In [ ]:
# check the matching between max_pred_celltype and max_pred_subtype

mix_col_after = pd.crosstab(adata_non_islet.obs['max_pred_celltype'], 
                      adata_non_islet.obs['max_pred_subtype'], normalize='columns')
mix_row_after = pd.crosstab(adata_non_islet.obs['max_pred_celltype'], 
                      adata_non_islet.obs['max_pred_subtype'], normalize='index')
mix_all_after = pd.crosstab(adata_non_islet.obs['max_pred_celltype'], 
                      adata_non_islet.obs['max_pred_subtype'], normalize='all')


fig, ax = plt.subplots(figsize=[7, 3])
sns.heatmap(mix_col_after, vmin=0, vmax=1, ax=ax)   # purity per subtype
# sns.heatmap(mix_row, vmin=0, vmax=1) # recall per coarse type
# sns.heatmap(mix_all, vmin=0, vmax=1) # global fractions


In [ ]:
expected = {
    # tweak as needed
    **{c: 'Acinar'     for c in mix_col_after.columns if 'acinar'     in c.lower()},
    **{c: 'Ductal'     for c in mix_col_after.columns if 'ductal'     in c.lower()},
    **{c: 'Endothelial'for c in mix_col_after.columns if 'ecs'        in c.lower()},
    **{c: 'Immune'     for c in mix_col_after.columns if any(k in c.lower() for k in ['macroph', 'plasma', 't_cells','mast'])},
    **{c: 'Stellate'   for c in mix_col_after.columns if 'stellate'   in c.lower()},
    **{c: 'Schwann'    for c in mix_col_after.columns if 'schwann'    in c.lower()},
}

threshold = 0.7  # purity cutoff
summary = []
for col in mix_col_after.columns:
    exp = expected.get(col)
    maj = mix_col_after[col].idxmax()
    exp_share = mix_col_after.loc[exp, col] if exp else float('nan')
    other = mix_col_after[col].drop(exp).idxmax() if exp else mix_col_after[col].idxmax()
    summary.append({
        "subtype": col,
        "expected": exp,
        "majority": maj,
        "purity_expected": exp_share,
        "top_offtarget": other,
        "flag_mislabeled": (exp is None) or (maj != exp) or (exp_share < threshold),
    })
summary = pd.DataFrame(summary).set_index("subtype").sort_values(["flag_mislabeled","purity_expected"])
summary

# Combine subtype annotation into the spatial data

In [ ]:
cst_c2l_abund_path = '../../data/cell2loc_others/cell2loc_non_islet_all_nonendocrine_subtypes/cst_c2l/non_islet_c2l_merged_cst_abundance.tsv'

cst_c2l = pd.read_csv(cst_c2l_abund_path, sep='\t', index_col=0)
cst_c2l = cst_c2l.rename(
    columns={
    'q05cell_abundance_w_sf_Pancreatic_persistent_memory_like_T_cells': 'q05cell_abundance_w_sf_T_cells',
    'Pancreatic_resident_macrophages': 'q05cell_abundance_w_sf_Macrophages',
    'q05cell_abundance_w_sf_CD27-_IgA+_plasmablasts': 'q05cell_abundance_w_sf_Plasmablasts',
})

tmp = pd.DataFrame(
    np.nan,
    index=adata.obs_names,
    columns=cst_c2l.columns
)

common_idx = adata.obs_names.intersection(cst_c2l.index)
tmp.loc[common_idx] = cst_c2l.loc[common_idx]

# store in obsm
adata.obsm["q05_cell_abundance_w_sf_subtype"] = tmp
# also label the c2l of cell type
adata.obsm["q05_cell_abundance_w_sf_celltype"] = adata.obsm.pop("q05_cell_abundance_w_sf")

In [ ]:
# create a new obs column 'annot_celltype' to store cell type anootation for analysis and visualization
adata.obs['annot_celltype'] = np.nan

# use the cell2location annotation from non islet non endocrine spots
common_idx = adata.obs_names.intersection(adata_non_islet.obs_names)
adata.obs.loc[common_idx, 'annot_celltype'] = (
    adata_non_islet.obs.loc[common_idx, 'max_pred_celltype']
)

# mark the islet spots as endocrine
mask_islet = adata.obs['Endocrine_type'].eq('islet_endocrine')
adata.obs.loc[mask_islet, 'annot_celltype'] = 'Islet_endocrine'

# for extra islet endocrine spots, only label trusted beta spots
mask_extra = (
    adata.obs['Endocrine_type'].eq('extra_endocrine') & 
    ((adata.obs['Endocrine'] - adata.obs['Acinar']) >= 0.2)
)
adata.obs.loc[mask_extra, 'annot_celltype'] = 'Extra_islet_endocrine'

# create a new obs column 'annot_subtype' to store cell subtype anootation
adata.obs['annot_cellsubtype'] = np.nan

# use the cell2location annotation from non islet non endocrine spots
common_idx = adata.obs_names.intersection(adata_non_islet.obs_names)
adata.obs.loc[common_idx, 'annot_cellsubtype'] = (
    adata_non_islet.obs.loc[common_idx, 'max_pred_subtype']
)

# for islet spots use the sharpened islet label
mask_islet = adata.obs['Endocrine_type'].eq('islet_endocrine')
adata.obs.loc[mask_islet, 'annot_cellsubtype'] = 'Islet_' + adata.obs.loc[mask_islet, 'islet_label_sharpen'].astype(str)

# for extra islet endocrine spots, only label trusted beta spots
mask_extra_beta = (
    adata.obs['Endocrine_type'].eq('extra_endocrine') & 
    ((adata.obs['Endocrine'] - adata.obs['Acinar']) >= 0.2) &
    (adata.obs['islet_label'].eq('β'))
)
adata.obs.loc[mask_extra_beta, 'annot_cellsubtype'] = 'Extra_islet_β'

mask_extra_others = (
    adata.obs['Endocrine_type'].eq('extra_endocrine') & 
    ((adata.obs['Endocrine'] - adata.obs['Acinar']) >= 0.2) &
    (adata.obs['islet_label'] != 'β')
)
adata.obs.loc[mask_extra_others, 'annot_cellsubtype'] = 'Extra_islet_non-β'

# save this annotation
adata.write('../../data/YK_raw_spatial_ct_cst_c2l_annot_hq_neighbor_clustering_20260107.h5ad')

# rename acinar and ductal subtypes

In [10]:
# add default colors for each cell type and cell subtype

def set_adata_colors(
    adata,
    col,
    *,
    color_map=None,          # dict: category -> color (hex or named)
):
    """
    Ensure adata.obs[col] is categorical and set adata.uns[f"{col}_colors"]
    with NO None values (Scanpy requires valid colors for all categories).
    """
    if color_map is None:
        color_map = {}

    # --- ensure categorical
    s = adata.obs[col]
    cats = list(adata.obs[col].cat.categories)

    # --- big fallback palette (repeatable)
    base = []
    for cm in ["tab20", "tab20b", "tab20c"]:
        base.extend([mpl.colors.to_hex(c) for c in mpl.cm.get_cmap(cm).colors])
    # repeat if needed
    fallback = (base * ((len(cats) // len(base)) + 1))[:len(cats)]

    # --- start from fallback then override with user map
    colors = fallback[:]
    for i, c in enumerate(cats):
        if c in color_map and color_map[c] is not None:
            colors[i] = mpl.colors.to_hex(mpl.colors.to_rgb(color_map[c]))

    # --- final safety: replace any accidental 'None' or invalid strings
    colors = ["#BFBFBF" if (x is None or str(x).lower() == "none") else x for x in colors]

    adata.uns[f"{col}_colors"] = colors
    return cats, colors

annot_colors = {
    'Acinar': (1.0, 0.831, 0.624),
    'Stellate': (0.8705882352941177, 0.5607843137254902, 0.0196078431372549), 
    'Ductal': (0.00392156862745098, 0.45098039215686275, 0.6980392156862745),
    'Endothelial': (0.5019607843137255, 0.0, 0.5019607843137255),
    'Islet_endocrine': (0.00784313725490196, 0.6196078431372549, 0.45098039215686275),
    'Extra_islet_endocrine': (1.0, 0.831, 0.624),
    
    'Immune': (0.984313725490196, 0.6862745098039216, 0.8941176470588236),
    'Schwann': (0.5019607843137255, 0.5019607843137255, 0.5019607843137255),

    'Activated_stellates': (1.0, 0.8941176470588236, 0.7686274509803922),
    'Quiescent_stellates': (1.0, 0.5490196078431373, 0.0),

    'Canonical_ductal': (0.6901960784313725, 0.7686274509803922, 0.8705882352941177),
    'Inflam_ductal_1': (0.0, 0.7490196078431373, 1.0), 
    'Inflam_ductal_2': (0.2549019607843137, 0.4117647058823529, 0.8823529411764706),
    'MUC5B+_ductal': (0.0, 0.0, 0.5019607843137255),

    'Endothelial': (0.5019607843137255, 0.0, 0.5019607843137255), 
    'Arterial_ECs': (0.8666666666666667, 0.6274509803921569, 0.8666666666666667),
    'Venous_ECs': (0.9333333333333333, 0.5098039215686274, 0.9333333333333333),
    'Capillary_ECs': (0.5019607843137255, 0.0, 0.5019607843137255),
    'Lymphatic_ECs': (0.29411764705882354, 0.0, 0.5098039215686274),

    'Islet_α': (0.5607843137254902, 0.7372549019607844, 0.5607843137254902),
    'Islet_β': (0.0, 0.5019607843137255, 0.0),
    'Islet_γ': (0.5647058823529412, 0.9333333333333333, 0.5647058823529412),
    'Islet_δ': (0.3333333333333333, 0.4196078431372549, 0.1843137254901961),
    'Islet_α_β': (0.2000, 0.6500, 0.3500), 

    'Extra_islet_β': (0.0000, 0.6500, 0.5500),
    'Extra_islet_non-β': (0.4000, 0.8000, 0.6500),

    'Macrophages': (0.8470588235294118, 0.7490196078431373, 0.8470588235294118),
    'Plasmablasts':  (1.0, 0.7529411764705882, 0.796078431372549),
    'T_cells': (0.8588235294117647, 0.4392156862745098, 0.5764705882352941),
    'Mast_cells': (1.0, 0.0, 1.0),

    'Schwann': (0.5019607843137255, 0.5019607843137255, 0.5019607843137255)
}

# reorder annotated cell types and subtyoes
adata.obs['annot_celltype'] = adata.obs['annot_celltype'].cat.reorder_categories(
    ['Islet_endocrine', 'Extra_islet_endocrine', 
     'Acinar', 'Ductal', 'Endothelial', 'Stellate', 
     'Immune', 'Schwann']
)
adata.obs['annot_cellsubtype'] = adata.obs['annot_cellsubtype'].cat.reorder_categories(
    ['Islet_α', 'Islet_β', 'Islet_α_β', 'Islet_γ', 'Islet_δ', 'Islet_Low quality',
     'Extra_islet_β', 'Extra_islet_non-β', 
     'Acinar', 
     'Canonical_ductal', 'Inflam_ductal_1', 'Inflam_ductal_2', 'MUC5B+_ductal', 
     'Arterial_ECs', 'Venous_ECs','Capillary_ECs', 'Lymphatic_ECs', 
     'Macrophages', 'Plasmablasts', 'T_cells', 'Mast_cells', 
     'Quiescent_stellates', 'Activated_stellates',  
     'Schwann']
)

In [11]:
# this dataset includes the Local Moran's I results
adata = sc.read_h5ad('../../data/YK_raw_spatial_ct_cst_c2l_annot_hq_neighbor_clustering_20260107.h5ad')

adata.obs['annot_cellsubtype'] = adata.obs['annot_cellsubtype'].astype(str)
adata.obs['annot_cellsubtype'] = adata.obs['annot_cellsubtype'].astype(str).replace({
    'Basal_ductal': 'Canonical_ductal'
})

# add build in colors of each cell type and cell subtype
set_adata_colors(adata, "annot_celltype", color_map=annot_colors)
set_adata_colors(adata, "annot_cellsubtype", color_map=annot_colors)

adata.write('../../data/YK_raw_spatial_ct_cst_c2l_annot_hq_neighbor_clustering_20260107.h5ad')

(['Islet_α',
  'Islet_β',
  'Islet_α_β',
  'Islet_γ',
  'Islet_δ',
  'Islet_Low quality',
  'Extra_islet_β',
  'Extra_islet_non-β',
  'Acinar',
  'Canonical_ductal',
  'Inflam_ductal_1',
  'Inflam_ductal_2',
  'MUC5B+_ductal',
  'Arterial_ECs',
  'Venous_ECs',
  'Capillary_ECs',
  'Lymphatic_ECs',
  'Macrophages',
  'Plasmablasts',
  'T_cells',
  'Mast_cells',
  'Quiescent_stellates',
  'Activated_stellates',
  'Schwann'],
 ['#8fbc8f',
  '#008000',
  '#33a659',
  '#90ee90',
  '#556b2f',
  '#98df8a',
  '#00a68c',
  '#66cca6',
  '#ffd49f',
  '#b0c4de',
  '#00bfff',
  '#4169e1',
  '#000080',
  '#dda0dd',
  '#ee82ee',
  '#800080',
  '#4b0082',
  '#d8bfd8',
  '#ffc0cb',
  '#db7093',
  '#ff00ff',
  '#ff8c00',
  '#ffe4c4',
  '#808080'])